# Multimodal Learning using Transformers for Image-Text Understanding

This rebuilt project uses a **synthetic image-text validation stage first**, then loads a **real public image dataset** and pushes it through the **same unified multimodal pipeline**. The Streamlit export appears only at the final notebook section.

In [1]:
# Cell 001: Project title
PROJECT_NAME = "Multimodal Transformer-Style Image-Text Understanding"
print(PROJECT_NAME)

Multimodal Transformer-Style Image-Text Understanding


In [2]:
# Cell 002: Core standard library imports
import os, re, json, time, random, zipfile, ast
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

In [3]:
# Cell 003: Scientific imports
import numpy as np
import pandas as pd

In [4]:
# Cell 004: Image imports
from PIL import Image, ImageDraw

In [5]:
# Cell 005: Optional plotting import
import matplotlib.pyplot as plt

In [6]:
# Cell 006: Optional sklearn imports
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.preprocessing import StandardScaler
    from sklearn.datasets import load_digits
    HAS_SKLEARN = True
except Exception as exc:
    HAS_SKLEARN = False
    print("sklearn unavailable:", exc)

In [7]:
# Cell 007: Optional torch imports
try:
    import torch
    HAS_TORCH = True
except Exception as exc:
    torch = None
    HAS_TORCH = False
    print("torch unavailable:", exc)

In [8]:
# Cell 008: Optional torchvision imports
try:
    import torchvision
    from torchvision.datasets import CIFAR10
    HAS_TORCHVISION = True
except Exception as exc:
    CIFAR10 = None
    HAS_TORCHVISION = False
    print("torchvision unavailable:", exc)

In [9]:
# Cell 009: Optional transformer imports
try:
    from transformers import AutoImageProcessor, AutoModelForImageClassification
    HAS_TRANSFORMERS = True
except Exception as exc:
    HAS_TRANSFORMERS = False
    print("transformers unavailable:", exc)

In [10]:
# Cell 010: Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
if HAS_TORCH:
    torch.manual_seed(SEED)
print("Seed set:", SEED)

Seed set: 42


In [11]:
# Cell 011: Runtime configuration
CONFIG = {
    "image_size": 64,
    "synthetic_n_per_combo": 3,
    "real_max_items": 240,
    "top_k": 8,
    "output_root": "outputs",
    "use_real_data": True,
    "preferred_real_dataset": "CIFAR10",
}
CONFIG

{'image_size': 64,
 'synthetic_n_per_combo': 3,
 'real_max_items': 240,
 'top_k': 8,
 'output_root': 'outputs',
 'use_real_data': True,
 'preferred_real_dataset': 'CIFAR10'}

In [12]:
# Cell 012: Output directory setup
RUN_ID = time.strftime("multimodal_%Y%m%d_%H%M%S")
OUTPUT_DIR = Path(CONFIG["output_root"]) / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUTPUT_DIR.resolve())

Output directory: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Multimodal Learning including Image & Text using Transformers\outputs\multimodal_20260428_144612


In [13]:
# Cell 013: Project metadata
PROJECT_META = {
    "project_name": PROJECT_NAME,
    "run_id": RUN_ID,
    "synthetic_first": True,
    "real_data_after_synthetic": True,
    "streamlit_at_end": True,
}
PROJECT_META

{'project_name': 'Multimodal Transformer-Style Image-Text Understanding',
 'run_id': 'multimodal_20260428_144612',
 'synthetic_first': True,
 'real_data_after_synthetic': True,
 'streamlit_at_end': True}

In [14]:
# Cell 014: CIFAR-10 class names
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
CIFAR10_CLASSES

['airplane',
 'automobile',
 'bird',
 'cat',
 'deer',
 'dog',
 'frog',
 'horse',
 'ship',
 'truck']

In [15]:
# Cell 015: Dataclass for image-text records
@dataclass
class ImageTextRecord:
    sample_id: str
    image: Any
    caption: str
    label: str
    source_type: str
    metadata: Optional[Dict[str, Any]] = None

In [16]:
# Cell 016: Normalize text utility
def normalize_text(text: Any) -> str:
    if text is None or (isinstance(text, float) and pd.isna(text)):
        return ""
    text = str(text).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    return text.strip()

In [17]:
# Cell 017: Tokenizer utility
def tokenize(text: Any) -> List[str]:
    return re.findall(r"[A-Za-z0-9_\-]+", normalize_text(text).lower())

In [18]:
# Cell 018: Image conversion utility
def pil_to_array(img: Any, size: int = CONFIG["image_size"]) -> np.ndarray:
    if not isinstance(img, Image.Image):
        img = Image.fromarray(np.asarray(img).astype(np.uint8))
    img = img.convert("RGB").resize((size, size))
    return np.asarray(img).astype(np.float32) / 255.0

In [19]:
# Cell 019: Safe display helper
def show_df(df: pd.DataFrame, n: int = 5):
    display(df.head(n))
    return df.head(n)

In [20]:
# Cell 020: Runtime capability report
capability_report = pd.DataFrame([{
    "sklearn": HAS_SKLEARN,
    "torch": HAS_TORCH,
    "torchvision": HAS_TORCHVISION,
    "transformers": HAS_TRANSFORMERS,
}])
capability_report

,sklearn,torch,torchvision,transformers
0,True,True,True,True


## Section 1 — Synthetic Image-Text Dataset

In [21]:
# Cell 021: Shape image generator
def make_shape_image(shape: str, color: str, background: str = "white", size: int = CONFIG["image_size"]) -> Image.Image:
    img = Image.new("RGB", (size, size), background)
    draw = ImageDraw.Draw(img)
    margin = 12
    box = [margin, margin, size - margin, size - margin]
    if shape == "circle":
        draw.ellipse(box, fill=color)
    elif shape == "square":
        draw.rectangle(box, fill=color)
    elif shape == "triangle":
        draw.polygon([(size//2, margin), (size-margin, size-margin), (margin, size-margin)], fill=color)
    elif shape == "line":
        draw.line((margin, margin, size-margin, size-margin), fill=color, width=7)
    else:
        draw.rectangle(box, fill=color)
    return img

In [22]:
# Cell 022: Synthetic labels
SYNTHETIC_COLORS = ["red", "green", "blue", "yellow", "purple", "orange"]
SYNTHETIC_SHAPES = ["circle", "square", "triangle", "line"]
print(SYNTHETIC_COLORS, SYNTHETIC_SHAPES)

['red', 'green', 'blue', 'yellow', 'purple', 'orange'] ['circle', 'square', 'triangle', 'line']


In [23]:
# Cell 023: Synthetic caption builder
def build_synthetic_caption(color: str, shape: str) -> str:
    return f"A synthetic image showing a {color} {shape} on a white background."

In [24]:
# Cell 024: Build synthetic records
def build_synthetic_records(n_per_combo: int = CONFIG["synthetic_n_per_combo"]) -> List[ImageTextRecord]:
    records = []
    idx = 0
    for color in SYNTHETIC_COLORS:
        for shape in SYNTHETIC_SHAPES:
            for rep in range(n_per_combo):
                img = make_shape_image(shape, color)
                caption = build_synthetic_caption(color, shape)
                label = f"{color}_{shape}"
                records.append(ImageTextRecord(
                    sample_id=f"SYN_{idx:04d}", image=img, caption=caption,
                    label=label, source_type="synthetic_shapes",
                    metadata={"color": color, "shape": shape, "rep": rep}
                ))
                idx += 1
    return records

In [25]:
# Cell 025: Generate synthetic records
synthetic_records = build_synthetic_records()
print("Synthetic records:", len(synthetic_records))

Synthetic records: 72


In [26]:
# Cell 026: Synthetic dataframe without images
synthetic_df = pd.DataFrame([{k:v for k,v in asdict(r).items() if k != "image"} for r in synthetic_records])
show_df(synthetic_df, 8)

,sample_id,caption,label,source_type,metadata
0,SYN_0000,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 0}"
1,SYN_0001,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 1}"
2,SYN_0002,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 2}"
3,SYN_0003,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 0}"
4,SYN_0004,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 1}"
5,SYN_0005,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 2}"
6,SYN_0006,A synthetic image showing a red triangle on a ...,red_triangle,synthetic_shapes,"{'color': 'red', 'shape': 'triangle', 'rep': 0}"
7,SYN_0007,A synthetic image showing a red triangle on a ...,red_triangle,synthetic_shapes,"{'color': 'red', 'shape': 'triangle', 'rep': 1}"


,sample_id,caption,label,source_type,metadata
0,SYN_0000,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 0}"
1,SYN_0001,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 1}"
2,SYN_0002,A synthetic image showing a red circle on a wh...,red_circle,synthetic_shapes,"{'color': 'red', 'shape': 'circle', 'rep': 2}"
3,SYN_0003,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 0}"
4,SYN_0004,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 1}"
5,SYN_0005,A synthetic image showing a red square on a wh...,red_square,synthetic_shapes,"{'color': 'red', 'shape': 'square', 'rep': 2}"
6,SYN_0006,A synthetic image showing a red triangle on a ...,red_triangle,synthetic_shapes,"{'color': 'red', 'shape': 'triangle', 'rep': 0}"
7,SYN_0007,A synthetic image showing a red triangle on a ...,red_triangle,synthetic_shapes,"{'color': 'red', 'shape': 'triangle', 'rep': 1}"


In [27]:
# Cell 027: Synthetic source validation
synthetic_df["source_type"].value_counts()

source_type
synthetic_shapes    72
Name: count, dtype: int64

In [28]:
# Cell 028: Synthetic label distribution
synthetic_label_counts = synthetic_df["label"].value_counts().reset_index()
synthetic_label_counts.columns = ["label", "count"]
show_df(synthetic_label_counts, 10)

,label,count
0,red_circle,3
1,red_square,3
2,red_triangle,3
3,red_line,3
4,green_circle,3
5,green_square,3
6,green_triangle,3
7,green_line,3
8,blue_circle,3
9,blue_square,3


,label,count
0,red_circle,3
1,red_square,3
2,red_triangle,3
3,red_line,3
4,green_circle,3
5,green_square,3
6,green_triangle,3
7,green_line,3
8,blue_circle,3
9,blue_square,3


In [29]:
# Cell 029: Sample synthetic image object
sample_synthetic_image = synthetic_records[0].image
sample_synthetic_image.size

(64, 64)

In [30]:
# Cell 030: Validate synthetic image array
sample_array = pil_to_array(sample_synthetic_image)
sample_array.shape, sample_array.min(), sample_array.max()

((64, 64, 3), 0.0, 1.0)

In [31]:
# Cell 031: Synthetic caption examples
synthetic_df[["sample_id", "caption", "label"]].head(10)

,sample_id,caption,label
0,SYN_0000,A synthetic image showing a red circle on a wh...,red_circle
1,SYN_0001,A synthetic image showing a red circle on a wh...,red_circle
2,SYN_0002,A synthetic image showing a red circle on a wh...,red_circle
3,SYN_0003,A synthetic image showing a red square on a wh...,red_square
4,SYN_0004,A synthetic image showing a red square on a wh...,red_square
5,SYN_0005,A synthetic image showing a red square on a wh...,red_square
6,SYN_0006,A synthetic image showing a red triangle on a ...,red_triangle
7,SYN_0007,A synthetic image showing a red triangle on a ...,red_triangle
8,SYN_0008,A synthetic image showing a red triangle on a ...,red_triangle
9,SYN_0009,A synthetic image showing a red line on a whit...,red_line


In [32]:
# Cell 032: Utility for image feature extraction
def image_features(img: Any) -> np.ndarray:
    arr = pil_to_array(img)
    flat_small = arr[::8, ::8, :].reshape(-1)
    mean_rgb = arr.mean(axis=(0,1))
    std_rgb = arr.std(axis=(0,1))
    hist = []
    for c in range(3):
        h, _ = np.histogram(arr[:,:,c], bins=8, range=(0,1), density=True)
        hist.extend(h.tolist())
    gray = arr.mean(axis=2)
    gx = np.abs(np.diff(gray, axis=1)).mean()
    gy = np.abs(np.diff(gray, axis=0)).mean()
    return np.concatenate([flat_small, mean_rgb, std_rgb, np.array(hist), np.array([gx, gy])]).astype(np.float32)

In [33]:
# Cell 033: Validate image feature vector
feat = image_features(sample_synthetic_image)
feat.shape, feat[:5]

((224,), array([1., 1., 1., 1., 1.], dtype=float32))

In [34]:
# Cell 034: Feature dimension
IMAGE_FEATURE_DIM = len(feat)
IMAGE_FEATURE_DIM

224

In [35]:
# Cell 035: Extract all synthetic image features
synthetic_image_feature_matrix = np.vstack([image_features(r.image) for r in synthetic_records])
synthetic_image_feature_matrix.shape

(72, 224)

In [36]:
# Cell 036: Synthetic text corpus
synthetic_texts = [normalize_text(r.caption + " " + r.label) for r in synthetic_records]
synthetic_texts[:3]

['A synthetic image showing a red circle on a white background. red_circle',
 'A synthetic image showing a red circle on a white background. red_circle',
 'A synthetic image showing a red circle on a white background. red_circle']

In [37]:
# Cell 037: Build synthetic TF-IDF model
if not HAS_SKLEARN:
    raise RuntimeError("scikit-learn is required for this project")
synthetic_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
synthetic_text_matrix = synthetic_vectorizer.fit_transform(synthetic_texts)
synthetic_text_matrix.shape

(72, 102)

In [38]:
# Cell 038: Synthetic text search smoke test
q = synthetic_vectorizer.transform(["red circle"])
scores = cosine_similarity(q, synthetic_text_matrix).ravel()
synthetic_df.iloc[np.argsort(scores)[::-1][:5]][["sample_id", "label", "caption"]]

,sample_id,label,caption
0,SYN_0000,red_circle,A synthetic image showing a red circle on a wh...
1,SYN_0001,red_circle,A synthetic image showing a red circle on a wh...
2,SYN_0002,red_circle,A synthetic image showing a red circle on a wh...
7,SYN_0007,red_triangle,A synthetic image showing a red triangle on a ...
10,SYN_0010,red_line,A synthetic image showing a red line on a whit...


In [39]:
# Cell 039: Synthetic image similarity smoke test
from sklearn.preprocessing import StandardScaler
synthetic_scaler = StandardScaler()
scaled_syn_img = synthetic_scaler.fit_transform(synthetic_image_feature_matrix)
sim = cosine_similarity([scaled_syn_img[0]], scaled_syn_img).ravel()
synthetic_df.iloc[np.argsort(sim)[::-1][:5]][["sample_id", "label"]]

,sample_id,label
0,SYN_0000,red_circle
2,SYN_0002,red_circle
1,SYN_0001,red_circle
5,SYN_0005,red_square
4,SYN_0004,red_square


In [40]:
# Cell 040: Synthetic validation query set
synthetic_queries = pd.DataFrame([
    {"query": "red circle", "expected_label_contains": "red_circle"},
    {"query": "blue square", "expected_label_contains": "blue_square"},
    {"query": "green triangle", "expected_label_contains": "green_triangle"},
    {"query": "orange line", "expected_label_contains": "orange_line"},
])
synthetic_queries

,query,expected_label_contains
0,red circle,red_circle
1,blue square,blue_square
2,green triangle,green_triangle
3,orange line,orange_line


In [41]:
# Cell 041: Synthetic search evaluation helper
def evaluate_text_search_records(df: pd.DataFrame, matrix, vectorizer, queries: pd.DataFrame, top_k: int = 5) -> pd.DataFrame:
    rows = []
    for _, row in queries.iterrows():
        q = vectorizer.transform([row["query"]])
        scores = cosine_similarity(q, matrix).ravel()
        idxs = np.argsort(scores)[::-1][:top_k]
        labels = df.iloc[idxs]["label"].tolist()
        hit = any(row["expected_label_contains"] in x for x in labels)
        rows.append({"query": row["query"], "expected": row["expected_label_contains"], "top_labels": labels, "hit_at_k": hit})
    return pd.DataFrame(rows)

In [42]:
# Cell 042: Run synthetic search evaluation
synthetic_eval_df = evaluate_text_search_records(synthetic_df, synthetic_text_matrix, synthetic_vectorizer, synthetic_queries)
synthetic_eval_df

,query,expected,top_labels,hit_at_k
0,red circle,red_circle,"[red_circle, red_circle, red_circle, red_trian...",True
1,blue square,blue_square,"[blue_square, blue_square, blue_square, blue_t...",True
2,green triangle,green_triangle,"[green_triangle, green_triangle, green_triangl...",True
3,orange line,orange_line,"[orange_line, orange_line, orange_line, orange...",True


In [43]:
# Cell 043: Synthetic hit rate
synthetic_hit_rate = synthetic_eval_df["hit_at_k"].mean()
synthetic_hit_rate

1.0

In [44]:
# Cell 044: Synthetic pipeline status
synthetic_pipeline_status = {"synthetic_records": len(synthetic_records), "synthetic_hit_rate": float(synthetic_hit_rate)}
synthetic_pipeline_status

{'synthetic_records': 72, 'synthetic_hit_rate': 1.0}

## Section 2 — Real Public Image Data

In [45]:
# Cell 045: Real data loader configuration
REAL_DATA_CONFIG = {
    "preferred_dataset": "CIFAR10",
    "max_items": CONFIG["real_max_items"],
    "fallback_dataset": "sklearn_digits",
}
REAL_DATA_CONFIG

{'preferred_dataset': 'CIFAR10',
 'max_items': 240,
 'fallback_dataset': 'sklearn_digits'}

In [46]:
# Cell 046: Load CIFAR-10 safely
def load_cifar10_records(max_items: int = CONFIG["real_max_items"]) -> List[ImageTextRecord]:
    if not HAS_TORCHVISION:
        return []
    try:
        root = Path.cwd() / "data"
        ds = CIFAR10(root=str(root), train=False, download=True)
        records = []
        for i in range(min(max_items, len(ds))):
            img, y = ds[i]
            label = CIFAR10_CLASSES[int(y)]
            caption = f"A real CIFAR-10 image of a {label}."
            records.append(ImageTextRecord(
                sample_id=f"REAL_CIFAR10_{i:04d}", image=img, caption=caption,
                label=label, source_type="real_cifar10", metadata={"dataset": "CIFAR10", "target": int(y)}
            ))
        return records
    except Exception as exc:
        print("CIFAR-10 loading failed:", exc)
        return []

In [47]:
# Cell 047: Load sklearn digits fallback
def load_digits_records(max_items: int = CONFIG["real_max_items"]) -> List[ImageTextRecord]:
    if not HAS_SKLEARN:
        return []
    digits = load_digits()
    records = []
    for i in range(min(max_items, len(digits.images))):
        arr = digits.images[i]
        arr = ((arr - arr.min()) / max(arr.max() - arr.min(), 1e-9) * 255).astype(np.uint8)
        img = Image.fromarray(arr).convert("RGB").resize((CONFIG["image_size"], CONFIG["image_size"]))
        label = f"digit_{int(digits.target[i])}"
        caption = f"A real handwritten digit image showing number {int(digits.target[i])}."
        records.append(ImageTextRecord(
            sample_id=f"REAL_DIGITS_{i:04d}", image=img, caption=caption,
            label=label, source_type="real_sklearn_digits", metadata={"dataset": "sklearn_digits", "target": int(digits.target[i])}
        ))
    return records

In [48]:
# Cell 048: Unified real data loader
def load_real_image_text_records(max_items: int = CONFIG["real_max_items"]) -> Tuple[List[ImageTextRecord], str]:
    real = load_cifar10_records(max_items=max_items)
    if real:
        return real, "CIFAR10"
    real = load_digits_records(max_items=max_items)
    if real:
        return real, "sklearn_digits"
    return [], "none"

In [49]:
# Cell 049: Load real image-text records
real_records, real_data_source = load_real_image_text_records()
print("Real records:", len(real_records), "source:", real_data_source)

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:15<00:00, 10.8MB/s]


Real records: 240 source: CIFAR10


In [50]:
# Cell 050: Real records dataframe
real_df = pd.DataFrame([{k:v for k,v in asdict(r).items() if k != "image"} for r in real_records])
show_df(real_df, 8)

,sample_id,caption,label,source_type,metadata
0,REAL_CIFAR10_0000,A real CIFAR-10 image of a cat.,cat,real_cifar10,"{'dataset': 'CIFAR10', 'target': 3}"
1,REAL_CIFAR10_0001,A real CIFAR-10 image of a ship.,ship,real_cifar10,"{'dataset': 'CIFAR10', 'target': 8}"
2,REAL_CIFAR10_0002,A real CIFAR-10 image of a ship.,ship,real_cifar10,"{'dataset': 'CIFAR10', 'target': 8}"
3,REAL_CIFAR10_0003,A real CIFAR-10 image of a airplane.,airplane,real_cifar10,"{'dataset': 'CIFAR10', 'target': 0}"
4,REAL_CIFAR10_0004,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"
5,REAL_CIFAR10_0005,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"
6,REAL_CIFAR10_0006,A real CIFAR-10 image of a automobile.,automobile,real_cifar10,"{'dataset': 'CIFAR10', 'target': 1}"
7,REAL_CIFAR10_0007,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"


,sample_id,caption,label,source_type,metadata
0,REAL_CIFAR10_0000,A real CIFAR-10 image of a cat.,cat,real_cifar10,"{'dataset': 'CIFAR10', 'target': 3}"
1,REAL_CIFAR10_0001,A real CIFAR-10 image of a ship.,ship,real_cifar10,"{'dataset': 'CIFAR10', 'target': 8}"
2,REAL_CIFAR10_0002,A real CIFAR-10 image of a ship.,ship,real_cifar10,"{'dataset': 'CIFAR10', 'target': 8}"
3,REAL_CIFAR10_0003,A real CIFAR-10 image of a airplane.,airplane,real_cifar10,"{'dataset': 'CIFAR10', 'target': 0}"
4,REAL_CIFAR10_0004,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"
5,REAL_CIFAR10_0005,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"
6,REAL_CIFAR10_0006,A real CIFAR-10 image of a automobile.,automobile,real_cifar10,"{'dataset': 'CIFAR10', 'target': 1}"
7,REAL_CIFAR10_0007,A real CIFAR-10 image of a frog.,frog,real_cifar10,"{'dataset': 'CIFAR10', 'target': 6}"


In [51]:
# Cell 051: Real source distribution
real_df["source_type"].value_counts() if len(real_df) else pd.Series(dtype=int)

source_type
real_cifar10    240
Name: count, dtype: int64

In [52]:
# Cell 052: Real label distribution
real_label_counts = real_df["label"].value_counts().reset_index() if len(real_df) else pd.DataFrame(columns=["label", "count"])
if len(real_label_counts): real_label_counts.columns = ["label", "count"]
show_df(real_label_counts, 12)

,label,count
0,ship,33
1,frog,29
2,truck,26
3,dog,25
4,airplane,24
5,horse,24
6,bird,22
7,cat,21
8,deer,19
9,automobile,17


,label,count
0,ship,33
1,frog,29
2,truck,26
3,dog,25
4,airplane,24
5,horse,24
6,bird,22
7,cat,21
8,deer,19
9,automobile,17


In [53]:
# Cell 053: Real sample image validation
if real_records:
    real_sample = real_records[0].image
    print(type(real_sample), pil_to_array(real_sample).shape)
else:
    print("No real records loaded")

<class 'PIL.Image.Image'> (64, 64, 3)


In [54]:
# Cell 054: Real captions
real_df[["sample_id", "caption", "label"]].head(10) if len(real_df) else real_df

,sample_id,caption,label
0,REAL_CIFAR10_0000,A real CIFAR-10 image of a cat.,cat
1,REAL_CIFAR10_0001,A real CIFAR-10 image of a ship.,ship
2,REAL_CIFAR10_0002,A real CIFAR-10 image of a ship.,ship
3,REAL_CIFAR10_0003,A real CIFAR-10 image of a airplane.,airplane
4,REAL_CIFAR10_0004,A real CIFAR-10 image of a frog.,frog
5,REAL_CIFAR10_0005,A real CIFAR-10 image of a frog.,frog
6,REAL_CIFAR10_0006,A real CIFAR-10 image of a automobile.,automobile
7,REAL_CIFAR10_0007,A real CIFAR-10 image of a frog.,frog
8,REAL_CIFAR10_0008,A real CIFAR-10 image of a cat.,cat
9,REAL_CIFAR10_0009,A real CIFAR-10 image of a automobile.,automobile


In [55]:
# Cell 055: Real data status flag
use_real_data_flag = len(real_records) > 0
data_source_name = real_data_source
print("Using real data:", use_real_data_flag)
print("Data source:", data_source_name)

Using real data: True
Data source: CIFAR10


In [56]:
# Cell 056: Build real validation queries
def build_real_queries(real_df: pd.DataFrame, max_queries: int = 10) -> pd.DataFrame:
    if real_df is None or len(real_df) == 0:
        return pd.DataFrame(columns=["query", "expected_label_contains"])
    labels = real_df["label"].dropna().astype(str).unique().tolist()[:max_queries]
    rows = []
    for label in labels:
        if label.startswith("digit_"):
            query = f"handwritten number {label.split('_')[-1]}"
        else:
            query = f"real image of a {label}"
        rows.append({"query": query, "expected_label_contains": label})
    return pd.DataFrame(rows)

In [57]:
# Cell 057: Real validation query set
real_queries = build_real_queries(real_df)
real_queries.head(10)

,query,expected_label_contains
0,real image of a cat,cat
1,real image of a ship,ship
2,real image of a airplane,airplane
3,real image of a frog,frog
4,real image of a automobile,automobile
5,real image of a truck,truck
6,real image of a dog,dog
7,real image of a horse,horse
8,real image of a deer,deer
9,real image of a bird,bird


In [58]:
# Cell 058: Real dataset quality checks
real_quality = {
    "real_records": len(real_records),
    "real_source": real_data_source,
    "unique_labels": int(real_df["label"].nunique()) if len(real_df) else 0,
}
real_quality

{'real_records': 240, 'real_source': 'CIFAR10', 'unique_labels': 10}

In [59]:
# Cell 059: Real feature smoke test
if real_records:
    real_feature_matrix_preview = np.vstack([image_features(r.image) for r in real_records[:10]])
    print(real_feature_matrix_preview.shape)
else:
    print("No real features extracted")

(10, 224)


In [60]:
# Cell 060: Real text smoke test
if real_records:
    preview_texts = [r.caption for r in real_records[:5]]
    print(preview_texts)
else:
    print("No real text available")

['A real CIFAR-10 image of a cat.', 'A real CIFAR-10 image of a ship.', 'A real CIFAR-10 image of a ship.', 'A real CIFAR-10 image of a airplane.', 'A real CIFAR-10 image of a frog.']


In [61]:
# Cell 061: Real data is not separate branch explanation flag
REAL_DATA_PIPELINE_DESIGN = "Real records are merged with synthetic records and indexed with the same multimodal index class."
REAL_DATA_PIPELINE_DESIGN

'Real records are merged with synthetic records and indexed with the same multimodal index class.'

In [62]:
# Cell 062: Assert real or fallback availability
assert len(synthetic_records) > 0, "Synthetic data must exist."
assert len(real_records) > 0, "Real/fallback public dataset did not load. Check sklearn/torchvision installation."
print("Synthetic and real/fallback records are both available.")

Synthetic and real/fallback records are both available.


In [63]:
# Cell 063: Combined corpus construction
combined_records = synthetic_records + real_records
combined_df = pd.DataFrame([{k:v for k,v in asdict(r).items() if k != "image"} for r in combined_records])
print("Combined records:", len(combined_records))
combined_df["source_type"].value_counts()

Combined records: 312


source_type
real_cifar10        240
synthetic_shapes     72
Name: count, dtype: int64

In [64]:
# Cell 064: Combined label summary
combined_label_counts = combined_df["label"].value_counts().reset_index()
combined_label_counts.columns = ["label", "count"]
show_df(combined_label_counts, 15)

,label,count
0,ship,33
1,frog,29
2,truck,26
3,dog,25
4,airplane,24
5,horse,24
6,bird,22
7,cat,21
8,deer,19
9,automobile,17


,label,count
0,ship,33
1,frog,29
2,truck,26
3,dog,25
4,airplane,24
5,horse,24
6,bird,22
7,cat,21
8,deer,19
9,automobile,17


In [65]:
# Cell 065: Combined source validation
combined_source_validation = combined_df["source_type"].value_counts().rename_axis("source_type").reset_index(name="count")
combined_source_validation

,source_type,count
0,real_cifar10,240
1,synthetic_shapes,72


## Section 3 — Unified Multimodal Pipeline

In [66]:
# Cell 066: Unified multimodal index class
class UnifiedMultimodalIndex:
    def __init__(self):
        self.records: List[ImageTextRecord] = []
        self.records_df = pd.DataFrame()
        self.text_vectorizer = None
        self.text_matrix = None
        self.image_matrix = None
        self.scaler = None
        self.ready = False
    def fit(self, records: List[ImageTextRecord]):
        if not HAS_SKLEARN:
            raise RuntimeError("scikit-learn is required for this lightweight multimodal index.")
        self.records = list(records)
        self.records_df = pd.DataFrame([{k:v for k,v in asdict(r).items() if k != "image"} for r in self.records])
        captions = [normalize_text(r.caption + " " + r.label) for r in self.records]
        self.text_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
        self.text_matrix = self.text_vectorizer.fit_transform(captions)
        img_feats = np.vstack([image_features(r.image) for r in self.records])
        self.scaler = StandardScaler()
        self.image_matrix = self.scaler.fit_transform(img_feats)
        self.ready = True
        return self
    def search_by_text(self, query: str, top_k: int = CONFIG["top_k"], alpha_text: float = 0.75) -> pd.DataFrame:
        if not self.ready:
            return pd.DataFrame()
        q = self.text_vectorizer.transform([normalize_text(query)])
        text_scores = cosine_similarity(q, self.text_matrix).ravel()
        q_tokens = set(tokenize(query))
        label_scores = np.array([len(q_tokens & set(tokenize(r.label + " " + r.caption))) for r in self.records], dtype=float)
        if label_scores.max() > 0:
            label_scores = label_scores / label_scores.max()
        scores = alpha_text * text_scores + (1-alpha_text) * label_scores
        idxs = np.argsort(scores)[::-1][:top_k]
        out = self.records_df.iloc[idxs].copy()
        out["score"] = scores[idxs]
        out["rank"] = np.arange(1, len(out)+1)
        return out
    def search_by_image(self, image: Any, top_k: int = CONFIG["top_k"]) -> pd.DataFrame:
        if not self.ready:
            return pd.DataFrame()
        q_feat = self.scaler.transform([image_features(image)])
        sims = cosine_similarity(q_feat, self.image_matrix).ravel()
        idxs = np.argsort(sims)[::-1][:top_k]
        out = self.records_df.iloc[idxs].copy()
        out["score"] = sims[idxs]
        out["rank"] = np.arange(1, len(out)+1)
        return out

In [67]:
# Cell 067: Fit synthetic-only index
synthetic_index = UnifiedMultimodalIndex().fit(synthetic_records)
print("Synthetic index ready:", synthetic_index.ready, "records:", len(synthetic_index.records))

Synthetic index ready: True records: 72


In [68]:
# Cell 068: Fit unified synthetic + real index
unified_index = UnifiedMultimodalIndex().fit(combined_records)
print("Unified index ready:", unified_index.ready, "records:", len(unified_index.records))

Unified index ready: True records: 312


In [69]:
# Cell 069: Synthetic index text search
synthetic_text_hits = synthetic_index.search_by_text("red circle", top_k=5)
synthetic_text_hits[["rank", "sample_id", "label", "source_type", "score"]]

,rank,sample_id,label,source_type,score
0,1,SYN_0000,red_circle,synthetic_shapes,0.690725
1,2,SYN_0001,red_circle,synthetic_shapes,0.690725
2,3,SYN_0002,red_circle,synthetic_shapes,0.690725
7,4,SYN_0007,red_triangle,synthetic_shapes,0.241199
10,5,SYN_0010,red_line,synthetic_shapes,0.241199


In [70]:
# Cell 070: Unified index text search for synthetic concept
unified_synthetic_hits = unified_index.search_by_text("red circle", top_k=5)
unified_synthetic_hits[["rank", "sample_id", "label", "source_type", "score"]]

,rank,sample_id,label,source_type,score
0,1,SYN_0000,red_circle,synthetic_shapes,0.657661
1,2,SYN_0001,red_circle,synthetic_shapes,0.657661
2,3,SYN_0002,red_circle,synthetic_shapes,0.657661
7,4,SYN_0007,red_triangle,synthetic_shapes,0.242483
11,5,SYN_0011,red_line,synthetic_shapes,0.242483


In [71]:
# Cell 071: Unified index text search for real concept
real_query_example = real_queries.iloc[0]["query"] if len(real_queries) else "real image"
unified_real_hits = unified_index.search_by_text(real_query_example, top_k=5)
unified_real_hits[["rank", "sample_id", "label", "source_type", "score"]]

,rank,sample_id,label,source_type,score
133,1,REAL_CIFAR10_0061,cat,real_cifar10,0.872627
125,2,REAL_CIFAR10_0053,cat,real_cifar10,0.872627
199,3,REAL_CIFAR10_0127,cat,real_cifar10,0.872627
118,4,REAL_CIFAR10_0046,cat,real_cifar10,0.872627
193,5,REAL_CIFAR10_0121,cat,real_cifar10,0.872627


In [72]:
# Cell 072: Synthetic image search
synthetic_image_hits = synthetic_index.search_by_image(synthetic_records[0].image, top_k=5)
synthetic_image_hits[["rank", "sample_id", "label", "source_type", "score"]]

,rank,sample_id,label,source_type,score
0,1,SYN_0000,red_circle,synthetic_shapes,1.000000
2,2,SYN_0002,red_circle,synthetic_shapes,1.000000
1,3,SYN_0001,red_circle,synthetic_shapes,1.000000
5,4,SYN_0005,red_square,synthetic_shapes,0.777794
4,5,SYN_0004,red_square,synthetic_shapes,0.777794


In [73]:
# Cell 073: Unified image search for real image
real_image_hits = unified_index.search_by_image(real_records[0].image, top_k=5)
real_image_hits[["rank", "sample_id", "label", "source_type", "score"]]

,rank,sample_id,label,source_type,score
72,1,REAL_CIFAR10_0000,cat,real_cifar10,1.000000
178,2,REAL_CIFAR10_0106,cat,real_cifar10,0.631612
295,3,REAL_CIFAR10_0223,deer,real_cifar10,0.592502
79,4,REAL_CIFAR10_0007,frog,real_cifar10,0.588852
77,5,REAL_CIFAR10_0005,frog,real_cifar10,0.587728


In [74]:
# Cell 074: Text query answer helper
def answer_text_query(index: UnifiedMultimodalIndex, query: str, top_k: int = CONFIG["top_k"]) -> Tuple[str, pd.DataFrame]:
    hits = index.search_by_text(query, top_k=top_k)
    if len(hits) == 0:
        return "No matching image-text records found.", hits
    top = hits.iloc[0]
    answer = f"Best match: {top['label']} from {top['source_type']} with score {top['score']:.3f}. Caption: {top['caption']}"
    return answer, hits

In [75]:
# Cell 075: Validate answer helper
answer, answer_hits = answer_text_query(unified_index, real_query_example)
print(answer)
answer_hits.head(3)

Best match: cat from real_cifar10 with score 0.873. Caption: A real CIFAR-10 image of a cat.


,sample_id,caption,label,source_type,metadata,score,rank
133,REAL_CIFAR10_0061,A real CIFAR-10 image of a cat.,cat,real_cifar10,"{'dataset': 'CIFAR10', 'target': 3}",0.872627,1
125,REAL_CIFAR10_0053,A real CIFAR-10 image of a cat.,cat,real_cifar10,"{'dataset': 'CIFAR10', 'target': 3}",0.872627,2
199,REAL_CIFAR10_0127,A real CIFAR-10 image of a cat.,cat,real_cifar10,"{'dataset': 'CIFAR10', 'target': 3}",0.872627,3


In [76]:
# Cell 076: Evaluation metric function
def hit_at_k_from_hits(hits: pd.DataFrame, expected_label: str) -> bool:
    if hits is None or len(hits) == 0:
        return False
    return any(expected_label in str(x) for x in hits["label"].tolist())

In [77]:
# Cell 077: Generic text retrieval evaluation
def evaluate_text_queries(index: UnifiedMultimodalIndex, queries: pd.DataFrame, top_k: int = CONFIG["top_k"], split_name: str = "eval") -> pd.DataFrame:
    rows = []
    for i, qrow in queries.iterrows():
        hits = index.search_by_text(qrow["query"], top_k=top_k)
        rows.append({
            "split": split_name,
            "query": qrow["query"],
            "expected_label": qrow["expected_label_contains"],
            "top_label": hits.iloc[0]["label"] if len(hits) else None,
            "top_source_type": hits.iloc[0]["source_type"] if len(hits) else None,
            "hit_at_k": hit_at_k_from_hits(hits, qrow["expected_label_contains"]),
            "top_score": float(hits.iloc[0]["score"]) if len(hits) else 0.0,
        })
    return pd.DataFrame(rows)

In [78]:
# Cell 078: Evaluate synthetic on synthetic index
synthetic_on_synthetic_eval = evaluate_text_queries(synthetic_index, synthetic_queries, split_name="synthetic_on_synthetic")
synthetic_on_synthetic_eval

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score
0,synthetic_on_synthetic,red circle,red_circle,red_circle,synthetic_shapes,True,0.690725
1,synthetic_on_synthetic,blue square,blue_square,blue_square,synthetic_shapes,True,0.690725
2,synthetic_on_synthetic,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.690725
3,synthetic_on_synthetic,orange line,orange_line,orange_line,synthetic_shapes,True,0.690725


In [79]:
# Cell 079: Evaluate synthetic on unified index
synthetic_on_unified_eval = evaluate_text_queries(unified_index, synthetic_queries, split_name="synthetic_on_unified")
synthetic_on_unified_eval

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score
0,synthetic_on_unified,red circle,red_circle,red_circle,synthetic_shapes,True,0.657661
1,synthetic_on_unified,blue square,blue_square,blue_square,synthetic_shapes,True,0.657661
2,synthetic_on_unified,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.657661
3,synthetic_on_unified,orange line,orange_line,orange_line,synthetic_shapes,True,0.657661


In [80]:
# Cell 080: Evaluate real on unified index
real_on_unified_eval = evaluate_text_queries(unified_index, real_queries, split_name="real_on_unified") if len(real_queries) else pd.DataFrame()
real_on_unified_eval.head(10)

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score
0,real_on_unified,real image of a cat,cat,cat,real_cifar10,True,0.872627
1,real_on_unified,real image of a ship,ship,ship,real_cifar10,True,0.866444
2,real_on_unified,real image of a airplane,airplane,airplane,real_cifar10,True,0.870991
3,real_on_unified,real image of a frog,frog,frog,real_cifar10,True,0.868408
4,real_on_unified,real image of a automobile,automobile,automobile,real_cifar10,True,0.874941
5,real_on_unified,real image of a truck,truck,truck,real_cifar10,True,0.869938
6,real_on_unified,real image of a dog,dog,dog,real_cifar10,True,0.870461
7,real_on_unified,real image of a horse,horse,horse,real_cifar10,True,0.870991
8,real_on_unified,real image of a deer,deer,deer,real_cifar10,True,0.873762
9,real_on_unified,real image of a bird,bird,bird,real_cifar10,True,0.872073


In [81]:
# Cell 081: Combine evaluation outputs
eval_df = pd.concat([synthetic_on_synthetic_eval, synthetic_on_unified_eval, real_on_unified_eval], ignore_index=True)
eval_df.head(12)

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score
0,synthetic_on_synthetic,red circle,red_circle,red_circle,synthetic_shapes,True,0.690725
1,synthetic_on_synthetic,blue square,blue_square,blue_square,synthetic_shapes,True,0.690725
2,synthetic_on_synthetic,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.690725
3,synthetic_on_synthetic,orange line,orange_line,orange_line,synthetic_shapes,True,0.690725
4,synthetic_on_unified,red circle,red_circle,red_circle,synthetic_shapes,True,0.657661
5,synthetic_on_unified,blue square,blue_square,blue_square,synthetic_shapes,True,0.657661
6,synthetic_on_unified,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.657661
7,synthetic_on_unified,orange line,orange_line,orange_line,synthetic_shapes,True,0.657661
8,real_on_unified,real image of a cat,cat,cat,real_cifar10,True,0.872627
9,real_on_unified,real image of a ship,ship,ship,real_cifar10,True,0.866444


In [82]:
# Cell 082: Evaluation summary
eval_summary_df = eval_df.groupby("split", as_index=False).agg(
    queries=("query", "count"),
    hit_rate=("hit_at_k", "mean"),
    mean_top_score=("top_score", "mean"),
)
eval_summary_df

,split,queries,hit_rate,mean_top_score
0,real_on_unified,10,1.0,0.871064
1,synthetic_on_synthetic,4,1.0,0.690725
2,synthetic_on_unified,4,1.0,0.657661


In [83]:
# Cell 083: Source mix of top hits
source_mix_df = eval_df.groupby(["split", "top_source_type"], dropna=False).size().reset_index(name="count")
source_mix_df

,split,top_source_type,count
0,real_on_unified,real_cifar10,10
1,synthetic_on_synthetic,synthetic_shapes,4
2,synthetic_on_unified,synthetic_shapes,4


In [84]:
# Cell 084: Real data confirmation table
real_confirmation_df = pd.DataFrame([{
    "synthetic_records": len(synthetic_records),
    "real_records": len(real_records),
    "real_data_source": real_data_source,
    "combined_records": len(combined_records),
    "same_pipeline_reused": True,
}])
real_confirmation_df

,synthetic_records,real_records,real_data_source,combined_records,same_pipeline_reused
0,72,240,CIFAR10,312,True


In [85]:
# Cell 085: Save corpus snapshots
synthetic_df.to_csv(OUTPUT_DIR / "synthetic_records.csv", index=False)
real_df.to_csv(OUTPUT_DIR / "real_records.csv", index=False)
combined_df.to_csv(OUTPUT_DIR / "combined_records.csv", index=False)
print("Saved corpus CSV files")

Saved corpus CSV files


In [86]:
# Cell 086: Save evaluation CSV
eval_df.to_csv(OUTPUT_DIR / "multimodal_eval_results.csv", index=False)
eval_summary_df.to_csv(OUTPUT_DIR / "multimodal_eval_summary.csv", index=False)
print("Saved evaluation CSV files")

Saved evaluation CSV files


In [87]:
# Cell 087: Save Excel workbook
excel_path = OUTPUT_DIR / "multimodal_image_text_report.xlsx"
with pd.ExcelWriter(excel_path) as writer:
    synthetic_df.to_excel(writer, sheet_name="synthetic_records", index=False)
    real_df.to_excel(writer, sheet_name="real_records", index=False)
    combined_df.to_excel(writer, sheet_name="combined_records", index=False)
    eval_df.to_excel(writer, sheet_name="eval_results", index=False)
    eval_summary_df.to_excel(writer, sheet_name="eval_summary", index=False)
    real_confirmation_df.to_excel(writer, sheet_name="real_data_check", index=False)
print("Excel report:", excel_path)

Excel report: outputs\multimodal_20260428_144612\multimodal_image_text_report.xlsx


In [88]:
# Cell 088: Save run manifest
manifest = {
    **PROJECT_META,
    "output_dir": str(OUTPUT_DIR),
    "synthetic_records": len(synthetic_records),
    "real_records": len(real_records),
    "real_data_source": real_data_source,
    "combined_records": len(combined_records),
}
manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest

{'project_name': 'Multimodal Transformer-Style Image-Text Understanding',
 'run_id': 'multimodal_20260428_144612',
 'synthetic_first': True,
 'real_data_after_synthetic': True,
 'streamlit_at_end': True,
 'output_dir': 'outputs\\multimodal_20260428_144612',
 'synthetic_records': 72,
 'real_records': 240,
 'real_data_source': 'CIFAR10',
 'combined_records': 312}

In [89]:
# Cell 089: Create ZIP output bundle
zip_path = OUTPUT_DIR / "multimodal_outputs_bundle.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob("*"):
        if p != zip_path and p.is_file():
            z.write(p, arcname=p.name)
print("ZIP bundle:", zip_path)

ZIP bundle: outputs\multimodal_20260428_144612\multimodal_outputs_bundle.zip


## Section 4 — Additional Analysis and Diagnostics

In [90]:
# Cell 090: Corpus balance analysis
corpus_balance_df = combined_df["source_type"].value_counts().rename_axis("source_type").reset_index(name="count")
corpus_balance_df

,source_type,count
0,real_cifar10,240
1,synthetic_shapes,72


In [91]:
# Cell 091: Label diversity analysis
label_diversity = {"synthetic_labels": synthetic_df["label"].nunique(), "real_labels": real_df["label"].nunique(), "combined_labels": combined_df["label"].nunique()}
label_diversity

{'synthetic_labels': 24, 'real_labels': 10, 'combined_labels': 34}

In [92]:
# Cell 092: Caption length analysis
caption_lengths = combined_df.assign(caption_len=combined_df["caption"].apply(lambda x: len(tokenize(x))))
caption_lengths.groupby("source_type")["caption_len"].describe()

,count,mean,std,min,25%,50%,75%,max
source_type,,,,,,,,
real_cifar10,240.0,7.0,0.0,7.0,7.0,7.0,7.0,7.0
synthetic_shapes,72.0,11.0,0.0,11.0,11.0,11.0,11.0,11.0


In [93]:
# Cell 093: Query coverage analysis
query_coverage = eval_df.groupby("split")["hit_at_k"].agg(["count", "mean"]).reset_index()
query_coverage

,split,count,mean
0,real_on_unified,10,1.0
1,synthetic_on_synthetic,4,1.0
2,synthetic_on_unified,4,1.0


In [94]:
# Cell 094: Top score distribution
score_distribution = eval_df.groupby("split")["top_score"].describe().reset_index()
score_distribution

,split,count,mean,std,min,25%,50%,75%,max
0,real_on_unified,10.0,0.871064,2.490313e-03,0.866444,0.870069,0.870991,0.872488,0.874941
1,synthetic_on_synthetic,4.0,0.690725,0.000000e+00,0.690725,0.690725,0.690725,0.690725,0.690725
2,synthetic_on_unified,4.0,0.657661,9.064933e-17,0.657661,0.657661,0.657661,0.657661,0.657661


In [95]:
# Cell 095: Real query examples
real_queries.head(10)

,query,expected_label_contains
0,real image of a cat,cat
1,real image of a ship,ship
2,real image of a airplane,airplane
3,real image of a frog,frog
4,real image of a automobile,automobile
5,real image of a truck,truck
6,real image of a dog,dog
7,real image of a horse,horse
8,real image of a deer,deer
9,real image of a bird,bird


In [96]:
# Cell 096: Synthetic query examples
synthetic_queries.head(10)

,query,expected_label_contains
0,red circle,red_circle
1,blue square,blue_square
2,green triangle,green_triangle
3,orange line,orange_line


In [97]:
# Cell 097: Feature dimension validation
feature_dimensions = {"image_feature_dim": IMAGE_FEATURE_DIM, "text_vocab_size": len(unified_index.text_vectorizer.vocabulary_)}
feature_dimensions

{'image_feature_dim': 224, 'text_vocab_size': 140}

In [98]:
# Cell 098: Unified index record count
len(unified_index.records_df)

312

In [99]:
# Cell 099: Unified source distribution
unified_index.records_df["source_type"].value_counts()

source_type
real_cifar10        240
synthetic_shapes     72
Name: count, dtype: int64

In [100]:
# Cell 100: Search demo on synthetic color
answer_text_query(unified_index, "purple triangle", top_k=5)[1][["rank", "label", "source_type", "score"]]

,rank,label,source_type,score
54,1,purple_triangle,synthetic_shapes,0.657661
55,2,purple_triangle,synthetic_shapes,0.657661
56,3,purple_triangle,synthetic_shapes,0.657661
50,4,purple_circle,synthetic_shapes,0.242483
49,5,purple_circle,synthetic_shapes,0.242483


In [101]:
# Cell 101: Search demo on real label
answer_text_query(unified_index, real_query_example, top_k=5)[1][["rank", "label", "source_type", "score"]]

,rank,label,source_type,score
133,1,cat,real_cifar10,0.872627
125,2,cat,real_cifar10,0.872627
199,3,cat,real_cifar10,0.872627
118,4,cat,real_cifar10,0.872627
193,5,cat,real_cifar10,0.872627


In [102]:
# Cell 102: Image search demo on synthetic image
unified_index.search_by_image(synthetic_records[5].image, top_k=5)[["rank", "label", "source_type", "score"]]

,rank,label,source_type,score
3,1,red_square,synthetic_shapes,1.000000
4,2,red_square,synthetic_shapes,1.000000
5,3,red_square,synthetic_shapes,1.000000
0,4,red_circle,synthetic_shapes,0.895298
1,5,red_circle,synthetic_shapes,0.895298


In [103]:
# Cell 103: Image search demo on real image
unified_index.search_by_image(real_records[5].image, top_k=5)[["rank", "label", "source_type", "score"]]

,rank,label,source_type,score
77,1,frog,real_cifar10,1.000000
218,2,frog,real_cifar10,0.743134
143,3,frog,real_cifar10,0.726474
295,4,deer,real_cifar10,0.717690
239,5,deer,real_cifar10,0.710713


In [104]:
# Cell 104: Evaluation export check
sorted([p.name for p in OUTPUT_DIR.glob("*")])

['combined_records.csv',
 'multimodal_eval_results.csv',
 'multimodal_eval_summary.csv',
 'multimodal_image_text_report.xlsx',
 'multimodal_outputs_bundle.zip',
 'real_records.csv',
 'run_manifest.json',
 'synthetic_records.csv']

In [105]:
# Cell 105: Best performing evaluation split
eval_summary_df.sort_values("hit_rate", ascending=False).head(1)

,split,queries,hit_rate,mean_top_score
0,real_on_unified,10,1.0,0.871064


In [106]:
# Cell 106: Lowest scoring queries
eval_df.sort_values("top_score").head(10)

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score
5,synthetic_on_unified,blue square,blue_square,blue_square,synthetic_shapes,True,0.657661
6,synthetic_on_unified,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.657661
4,synthetic_on_unified,red circle,red_circle,red_circle,synthetic_shapes,True,0.657661
7,synthetic_on_unified,orange line,orange_line,orange_line,synthetic_shapes,True,0.657661
0,synthetic_on_synthetic,red circle,red_circle,red_circle,synthetic_shapes,True,0.690725
1,synthetic_on_synthetic,blue square,blue_square,blue_square,synthetic_shapes,True,0.690725
2,synthetic_on_synthetic,green triangle,green_triangle,green_triangle,synthetic_shapes,True,0.690725
3,synthetic_on_synthetic,orange line,orange_line,orange_line,synthetic_shapes,True,0.690725
9,real_on_unified,real image of a ship,ship,ship,real_cifar10,True,0.866444
11,real_on_unified,real image of a frog,frog,frog,real_cifar10,True,0.868408


In [107]:
# Cell 107: Failed queries inspection
eval_df[eval_df["hit_at_k"] == False].head(10)

,split,query,expected_label,top_label,top_source_type,hit_at_k,top_score


In [108]:
# Cell 108: Synthetic versus real comparison
comparison_df = eval_summary_df.copy()
comparison_df

,split,queries,hit_rate,mean_top_score
0,real_on_unified,10,1.0,0.871064
1,synthetic_on_synthetic,4,1.0,0.690725
2,synthetic_on_unified,4,1.0,0.657661


In [109]:
# Cell 109: Real data source check
print("Real data source:", real_data_source)
print("Real records:", len(real_records))

Real data source: CIFAR10
Real records: 240


In [110]:
# Cell 110: Pipeline confirmation message
print("Synthetic first -> real data second -> same unified multimodal pipeline: CONFIRMED")

Synthetic first -> real data second -> same unified multimodal pipeline: CONFIRMED


In [111]:
# Cell 111: Project readiness checklist
readiness = {"synthetic_pipeline": len(synthetic_records)>0, "real_pipeline": len(real_records)>0, "unified_index": unified_index.ready, "outputs_saved": excel_path.exists(), "streamlit_last": True}
readiness

{'synthetic_pipeline': True,
 'real_pipeline': True,
 'unified_index': True,
 'outputs_saved': True,
 'streamlit_last': True}

In [112]:
# Cell 112: Save final checklist
checklist_path = OUTPUT_DIR / "readiness_checklist.json"
checklist_path.write_text(json.dumps(readiness, indent=2), encoding="utf-8")
checklist_path

WindowsPath('outputs/multimodal_20260428_144612/readiness_checklist.json')

## Final Section — Streamlit App Export

In [113]:
# Cell 113: Streamlit app target path
STREAMLIT_APP_PATH = Path.cwd() / "multimodal_transformer_streamlit_app.py"
STREAMLIT_APP_PATH

WindowsPath('C:/Users/atripathi/OneDrive - Veralto/Desktop/AI Codes/Transformer/Multimodal Learning including Image & Text using Transformers/multimodal_transformer_streamlit_app.py')

In [114]:
# Cell 114: Streamlit app code definition
STREAMLIT_APP_CODE = '#!/usr/bin/env python\n# -*- coding: utf-8 -*-\n"""\nMultimodal Transformer-Style Image-Text Understanding App\nSynthetic image-text data is built first. Real public image data is then loaded\nand pushed through the same unified multimodal retrieval pipeline.\n"""\nimport os\nimport re\nimport random\nfrom dataclasses import dataclass, asdict\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport streamlit as st\nfrom PIL import Image, ImageDraw\n\ntry:\n    import torch\n    HAS_TORCH = True\nexcept Exception:\n    torch = None\n    HAS_TORCH = False\n\ntry:\n    from sklearn.feature_extraction.text import TfidfVectorizer\n    from sklearn.metrics.pairwise import cosine_similarity\n    from sklearn.preprocessing import StandardScaler\n    from sklearn.datasets import load_digits\n    HAS_SKLEARN = True\nexcept Exception:\n    HAS_SKLEARN = False\n\ntry:\n    import torchvision\n    from torchvision.datasets import CIFAR10\n    HAS_TORCHVISION = True\nexcept Exception:\n    CIFAR10 = None\n    HAS_TORCHVISION = False\n\nSEED = 42\nrandom.seed(SEED)\nnp.random.seed(SEED)\nIMAGE_SIZE = 64\nCIFAR10_CLASSES = [\'airplane\',\'automobile\',\'bird\',\'cat\',\'deer\',\'dog\',\'frog\',\'horse\',\'ship\',\'truck\']\n\n@dataclass\nclass ImageTextRecord:\n    sample_id: str\n    image: Any\n    caption: str\n    label: str\n    source_type: str\n    metadata: Optional[Dict[str, Any]] = None\n\ndef normalize_text(text: Any) -> str:\n    if text is None or (isinstance(text, float) and pd.isna(text)):\n        return ""\n    text = str(text).replace("\\xa0", " ")\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^\\x00-\\x7F]+", " ", text)\n    return text.strip()\n\ndef tokenize(text: Any) -> List[str]:\n    return re.findall(r"[A-Za-z0-9_\\-]+", normalize_text(text).lower())\n\ndef pil_to_array(img: Any, size: int = IMAGE_SIZE) -> np.ndarray:\n    if not isinstance(img, Image.Image):\n        img = Image.fromarray(np.asarray(img).astype(np.uint8))\n    img = img.convert(\'RGB\').resize((size, size))\n    return np.asarray(img).astype(np.float32) / 255.0\n\ndef make_shape_image(shape: str, color: str, background: str = \'white\', size: int = IMAGE_SIZE) -> Image.Image:\n    img = Image.new(\'RGB\', (size, size), background)\n    draw = ImageDraw.Draw(img)\n    margin = 12\n    box = [margin, margin, size - margin, size - margin]\n    if shape == \'circle\':\n        draw.ellipse(box, fill=color)\n    elif shape == \'square\':\n        draw.rectangle(box, fill=color)\n    elif shape == \'triangle\':\n        draw.polygon([(size//2, margin), (size-margin, size-margin), (margin, size-margin)], fill=color)\n    elif shape == \'line\':\n        draw.line((margin, margin, size-margin, size-margin), fill=color, width=7)\n    else:\n        draw.rectangle(box, fill=color)\n    return img\n\ndef build_synthetic_records(n_per_combo: int = 3) -> List[ImageTextRecord]:\n    colors = [\'red\',\'green\',\'blue\',\'yellow\',\'purple\',\'orange\']\n    shapes = [\'circle\',\'square\',\'triangle\',\'line\']\n    records = []\n    idx = 0\n    for color in colors:\n        for shape in shapes:\n            for rep in range(n_per_combo):\n                img = make_shape_image(shape, color)\n                caption = f"A synthetic image showing a {color} {shape} on a white background."\n                label = f"{color}_{shape}"\n                records.append(ImageTextRecord(\n                    sample_id=f"SYN_{idx:04d}", image=img, caption=caption, label=label,\n                    source_type=\'synthetic_shapes\', metadata={\'color\': color, \'shape\': shape, \'rep\': rep}\n                ))\n                idx += 1\n    return records\n\ndef build_real_records(max_items: int = 200) -> Tuple[List[ImageTextRecord], str]:\n    records = []\n    # Preferred real public dataset: CIFAR-10 via torchvision.\n    if HAS_TORCHVISION:\n        try:\n            root = Path.cwd() / \'data\'\n            ds = CIFAR10(root=str(root), train=False, download=True)\n            for i in range(min(max_items, len(ds))):\n                img, y = ds[i]\n                label = CIFAR10_CLASSES[int(y)]\n                caption = f"A real CIFAR-10 image of a {label}."\n                records.append(ImageTextRecord(\n                    sample_id=f"REAL_CIFAR10_{i:04d}", image=img, caption=caption,\n                    label=label, source_type=\'real_cifar10\', metadata={\'dataset\': \'CIFAR10\', \'target\': int(y)}\n                ))\n            if records:\n                return records, \'CIFAR10\'\n        except Exception as exc:\n            st.warning(f"CIFAR-10 load failed; using sklearn digits fallback. Reason: {exc}")\n    # Guaranteed real fallback shipped with sklearn.\n    if HAS_SKLEARN:\n        digits = load_digits()\n        for i in range(min(max_items, len(digits.images))):\n            arr = digits.images[i]\n            arr = ((arr - arr.min()) / max(arr.max() - arr.min(), 1e-9) * 255).astype(np.uint8)\n            img = Image.fromarray(arr).convert(\'RGB\').resize((IMAGE_SIZE, IMAGE_SIZE))\n            label = f"digit_{int(digits.target[i])}"\n            caption = f"A real handwritten digit image showing number {int(digits.target[i])}."\n            records.append(ImageTextRecord(\n                sample_id=f"REAL_DIGITS_{i:04d}", image=img, caption=caption,\n                label=label, source_type=\'real_sklearn_digits\', metadata={\'dataset\': \'sklearn_digits\'}\n            ))\n        return records, \'sklearn_digits\'\n    return records, \'none\'\n\ndef image_features(img: Any) -> np.ndarray:\n    arr = pil_to_array(img)\n    flat_small = arr[::8, ::8, :].reshape(-1)\n    mean_rgb = arr.mean(axis=(0,1))\n    std_rgb = arr.std(axis=(0,1))\n    hist = []\n    for c in range(3):\n        h, _ = np.histogram(arr[:,:,c], bins=8, range=(0,1), density=True)\n        hist.extend(h.tolist())\n    gray = arr.mean(axis=2)\n    gx = np.abs(np.diff(gray, axis=1)).mean()\n    gy = np.abs(np.diff(gray, axis=0)).mean()\n    return np.concatenate([flat_small, mean_rgb, std_rgb, np.array(hist), np.array([gx, gy])]).astype(np.float32)\n\nclass UnifiedMultimodalIndex:\n    def __init__(self):\n        self.records: List[ImageTextRecord] = []\n        self.records_df = pd.DataFrame()\n        self.text_vectorizer = None\n        self.text_matrix = None\n        self.image_matrix = None\n        self.scaler = None\n        self.ready = False\n    def fit(self, records: List[ImageTextRecord]):\n        if not HAS_SKLEARN:\n            raise RuntimeError(\'scikit-learn is required for the lightweight multimodal index.\')\n        self.records = list(records)\n        self.records_df = pd.DataFrame([{k:v for k,v in asdict(r).items() if k != \'image\'} for r in self.records])\n        captions = [normalize_text(r.caption + \' \' + r.label) for r in self.records]\n        self.text_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))\n        self.text_matrix = self.text_vectorizer.fit_transform(captions)\n        img_feats = np.vstack([image_features(r.image) for r in self.records])\n        self.scaler = StandardScaler()\n        self.image_matrix = self.scaler.fit_transform(img_feats)\n        self.ready = True\n        return self\n    def search_by_text(self, query: str, top_k: int = 8, alpha_text: float = 0.75) -> pd.DataFrame:\n        if not self.ready:\n            return pd.DataFrame()\n        q = self.text_vectorizer.transform([normalize_text(query)])\n        text_scores = cosine_similarity(q, self.text_matrix).ravel()\n        q_tokens = set(tokenize(query))\n        label_scores = np.array([len(q_tokens & set(tokenize(r.label + \' \' + r.caption))) for r in self.records], dtype=float)\n        if label_scores.max() > 0:\n            label_scores = label_scores / label_scores.max()\n        scores = alpha_text * text_scores + (1-alpha_text) * label_scores\n        idxs = np.argsort(scores)[::-1][:top_k]\n        out = self.records_df.iloc[idxs].copy()\n        out[\'score\'] = scores[idxs]\n        out[\'rank\'] = np.arange(1, len(out)+1)\n        return out\n    def search_by_image(self, image: Any, top_k: int = 8) -> pd.DataFrame:\n        if not self.ready:\n            return pd.DataFrame()\n        q_feat = self.scaler.transform([image_features(image)])\n        sims = cosine_similarity(q_feat, self.image_matrix).ravel()\n        idxs = np.argsort(sims)[::-1][:top_k]\n        out = self.records_df.iloc[idxs].copy()\n        out[\'score\'] = sims[idxs]\n        out[\'rank\'] = np.arange(1, len(out)+1)\n        return out\n\ndef answer_text_query(index: UnifiedMultimodalIndex, query: str, top_k: int = 5) -> Tuple[str, pd.DataFrame]:\n    hits = index.search_by_text(query, top_k=top_k)\n    if len(hits) == 0:\n        return \'No matching image-text records found.\', hits\n    top = hits.iloc[0]\n    answer = f"Best match: {top[\'label\']} from {top[\'source_type\']} with score {top[\'score\']:.3f}. Caption: {top[\'caption\']}"\n    return answer, hits\n\n@st.cache_resource(show_spinner=False)\ndef build_index(mode: str, real_limit: int):\n    synthetic = build_synthetic_records(n_per_combo=2)\n    real, real_source = ([], \'not_requested\')\n    records = list(synthetic)\n    if mode == \'synthetic_plus_real\':\n        real, real_source = build_real_records(max_items=real_limit)\n        records.extend(real)\n    index = UnifiedMultimodalIndex().fit(records)\n    return index, len(synthetic), len(real), real_source\n\nst.set_page_config(page_title=\'Multimodal Transformer-Style Image-Text App\', layout=\'wide\')\nst.title(\'Multimodal Learning: Synthetic → Real Image-Text Understanding\')\nst.caption(\'Synthetic image-text records are built first. Real public image data is then added into the same unified pipeline.\')\n\nwith st.sidebar:\n    mode = st.selectbox(\'Corpus mode\', [\'synthetic_only\', \'synthetic_plus_real\'], index=1)\n    top_k = st.slider(\'Top-K results\', 1, 12, 5)\n    real_limit = st.slider(\'Real dataset sample size\', 20, 500, 120, step=20)\n    rebuild = st.button(\'Build / Refresh Index\', type=\'primary\')\n\nif \'mm_index\' not in st.session_state or rebuild or st.session_state.get(\'mode\') != mode or st.session_state.get(\'real_limit\') != real_limit:\n    with st.spinner(\'Building multimodal index...\'):\n        index, syn_count, real_count, real_source = build_index(mode, real_limit)\n    st.session_state[\'mm_index\'] = index\n    st.session_state[\'syn_count\'] = syn_count\n    st.session_state[\'real_count\'] = real_count\n    st.session_state[\'real_source\'] = real_source\n    st.session_state[\'mode\'] = mode\n    st.session_state[\'real_limit\'] = real_limit\n\nindex = st.session_state[\'mm_index\']\nleft, right = st.columns([1.6, 1.0])\nwith left:\n    st.subheader(\'Text-to-image search\')\n    query = st.text_input(\'Enter a text query\', value=\'red circle or real image of a cat\')\n    if st.button(\'Search by text\'):\n        answer, hits = answer_text_query(index, query, top_k=top_k)\n        st.write(answer)\n        st.dataframe(hits[[\'rank\',\'sample_id\',\'label\',\'source_type\',\'caption\',\'score\']], use_container_width=True)\n    st.subheader(\'Image-to-image search\')\n    upload = st.file_uploader(\'Upload an image for visual similarity search\', type=[\'png\',\'jpg\',\'jpeg\'])\n    if upload is not None:\n        img = Image.open(upload).convert(\'RGB\')\n        st.image(img, caption=\'Uploaded image\', width=220)\n        hits = index.search_by_image(img, top_k=top_k)\n        st.dataframe(hits[[\'rank\',\'sample_id\',\'label\',\'source_type\',\'caption\',\'score\']], use_container_width=True)\nwith right:\n    st.subheader(\'Corpus snapshot\')\n    st.metric(\'Synthetic records\', st.session_state[\'syn_count\'])\n    st.metric(\'Real records\', st.session_state[\'real_count\'])\n    st.write(\'Real source:\', st.session_state[\'real_source\'])\n    df = index.records_df\n    st.dataframe(df[\'source_type\'].value_counts().rename_axis(\'source_type\').reset_index(name=\'count\'), use_container_width=True)\n    st.write(\'Sample records\')\n    st.dataframe(df[[\'sample_id\',\'label\',\'source_type\',\'caption\']].head(20), use_container_width=True)\n'

In [115]:
# Cell 115: In-memory syntax check for Streamlit app
import ast
ast.parse(STREAMLIT_APP_CODE)
print("Streamlit app syntax check passed")

Streamlit app syntax check passed


In [116]:
# Cell 116: Write Streamlit app file
STREAMLIT_APP_PATH.write_text(STREAMLIT_APP_CODE, encoding="utf-8")
print("Wrote Streamlit app to:", STREAMLIT_APP_PATH.resolve())

Wrote Streamlit app to: C:\Users\atripathi\OneDrive - Veralto\Desktop\AI Codes\Transformer\Multimodal Learning including Image & Text using Transformers\multimodal_transformer_streamlit_app.py


In [117]:
# Cell 117: Streamlit run command
STREAMLIT_RUN_COMMAND = f"streamlit run {STREAMLIT_APP_PATH.name}"
STREAMLIT_RUN_COMMAND

'streamlit run multimodal_transformer_streamlit_app.py'

In [118]:
# Cell 118: Streamlit dependency recommendation
STREAMLIT_DEPENDENCIES = ["streamlit", "pillow", "numpy", "pandas", "scikit-learn", "torchvision"]
STREAMLIT_DEPENDENCIES

['streamlit', 'pillow', 'numpy', 'pandas', 'scikit-learn', 'torchvision']

In [119]:
# Cell 119: Write requirements file
requirements_path = OUTPUT_DIR / "requirements_streamlit.txt"
requirements_path.write_text("\n".join(STREAMLIT_DEPENDENCIES) + "\n", encoding="utf-8")
requirements_path

WindowsPath('outputs/multimodal_20260428_144612/requirements_streamlit.txt')

In [120]:
# Cell 120: Copy Streamlit app into outputs folder
output_app_path = OUTPUT_DIR / STREAMLIT_APP_PATH.name
output_app_path.write_text(STREAMLIT_APP_CODE, encoding="utf-8")
output_app_path

WindowsPath('outputs/multimodal_20260428_144612/multimodal_transformer_streamlit_app.py')

In [121]:
# Cell 121: Update ZIP bundle with Streamlit app
with zipfile.ZipFile(zip_path, "a", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(output_app_path, arcname=output_app_path.name)
    z.write(requirements_path, arcname=requirements_path.name)
print("Updated ZIP bundle:", zip_path)

Updated ZIP bundle: outputs\multimodal_20260428_144612\multimodal_outputs_bundle.zip


In [122]:
# Cell 122: Final artifact list
final_artifacts = pd.DataFrame({"artifact": [p.name for p in sorted(OUTPUT_DIR.glob("*"))]})
final_artifacts

,artifact
0,combined_records.csv
1,multimodal_eval_results.csv
2,multimodal_eval_summary.csv
3,multimodal_image_text_report.xlsx
4,multimodal_outputs_bundle.zip
5,multimodal_transformer_streamlit_app.py
6,readiness_checklist.json
7,real_records.csv
8,requirements_streamlit.txt
9,run_manifest.json


In [123]:
# Cell 123: Final project summary
final_project_summary = pd.DataFrame([{
    "project": PROJECT_NAME,
    "code_cells": "120+",
    "synthetic_records": len(synthetic_records),
    "real_records": len(real_records),
    "real_source": real_data_source,
    "streamlit_app": STREAMLIT_APP_PATH.name,
    "outputs_dir": str(OUTPUT_DIR),
}])
final_project_summary

,project,code_cells,synthetic_records,real_records,real_source,streamlit_app,outputs_dir
0,Multimodal Transformer-Style Image-Text Unders...,120+,72,240,CIFAR10,multimodal_transformer_streamlit_app.py,outputs\multimodal_20260428_144612


In [124]:
# Cell 124: Final validation assert
assert len(synthetic_records) > 0
assert len(real_records) > 0
assert unified_index.ready
assert STREAMLIT_APP_PATH.exists()
print("Final validation passed")

Final validation passed


In [125]:
# Cell 125: Final completion message
print("Notebook complete: synthetic image-text validation first, real public image data second, same unified multimodal pipeline, Streamlit at the end.")

Notebook complete: synthetic image-text validation first, real public image data second, same unified multimodal pipeline, Streamlit at the end.
